# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane:** Refresh / Content Opportunity Scoring

**Why this lane:** I'm choosing Refresh / Content Opportunity Scoring because it directly answers a practical business question: "Which pages should a content editor review first?" This lane produces a ranked action queue with reason codes, which is actionable and measurable.

The starter pipeline already provides a working template for this lane – I've run it and seen that a learned model (random forest) can achieve a Precision@50 of 0.740, meaning ~37 of the top 50 pages flagged for review actually show declining trends. This beats the hand-written rule baseline which only gets ~12 of 50 right. This is strong evidence that ML can add value beyond simple rules.

I'm also interested in the reason-code output – it makes the recommendations transparent and reviewable, which is critical for a decision-support tool. This aligns with my principle of building interpretable, evidence-based solutions rather than black-box models.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**What decision does this improve?**

This work improves the decision of **which content pages to review first** for refresh, update, or optimization. Instead of an editor manually scanning through hundreds or thousands of pages, the system produces a **ranked queue** – pages at the top are the ones most likely to benefit from attention.

This is a prioritization problem, not a classification problem. The goal is not to label every page as "good" or "bad," but to help a human decide where to invest their limited time first.

---

**Who acts on the output, and what do they do?**

A **content editor, SEO specialist, or site manager** acts on the output. They open the ranked queue, inspect the top 20-50 pages, and decide what action to take:

- **Refresh** – update the content with new information
- **Expand** – add more depth or sections
- **Consolidate** – merge with another page that covers similar topics
- **Protect** – monitor but take no immediate action
- **Prune** – remove or no-index the page

The reason codes provide transparency – they explain *why* each page was flagged, so the editor can trust the recommendation or override it if context suggests otherwise.

---

**What does a wrong recommendation cost?**

A wrong recommendation has two types of cost:

1. **False positive** (recommending a page that doesn't need review): Wastes editor time – each page reviewed takes 5-15 minutes. If the queue has too many false positives, the editor loses trust in the system and may stop using it. Over time, this undermines the tool's value.

2. **False negative** (missing a page that needs review): A declining page goes unnoticed, and its traffic continues to drop. Over time, this means lost visibility, lost clicks, and lost revenue. This cost is harder to measure but potentially larger.

Between the two, missing a truly declining page (false negative) is **more expensive** than wasting a few minutes on a false positive. This suggests we should favor recall (catching more true problems) over precision, especially in the top of the queue. However, the exact trade-off depends on the editor's capacity and the business context – this is a policy choice we can adjust.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    print("Working dir:", os.getcwd())

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "Starter CSV not found"
print("✅ Data loaded successfully.")

Working dir: /content/flyrank-ml-internship-starter
✅ Data loaded successfully.


In [2]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("=" * 60)
print("SUPPORTING EVIDENCE FOR LANE 2: REFRESH / CONTENT OPPORTUNITY SCORING")
print("=" * 60)

# Number 1: Model vs Baseline performance (from the pipeline you ran)
print("\n1. Learned model beats hand-written rule:")
print("   - Baseline (hand-written rule) Precision@50: 0.240 (~12 of top 50 right)")
print("   - Random Forest Precision@50: 0.740 (~37 of top 50 right)")
print("   → The learned model is ~3.1x more accurate at identifying pages worth reviewing.")

# Number 2: CTR collapses by position (from Discovery B)
print("\n2. CTR drops sharply as position worsens:")
ctr_by_pos = df[df["impressions_90d"] >= 100].groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
for tier, ctr in ctr_by_pos.round(4).items():
    print(f"   - {tier}: {ctr}% CTR")
print("   → Position is a strong signal for engagement opportunity.")

# Number 3: Word count is NOT the lever (from Discovery C)
print("\n3. Content length alone doesn't explain decline:")
wc_by_trend = df.groupby("trend_direction")["word_count"].median().round(0)
print(f"   - Declining pages (down): median {wc_by_trend['down']:.0f} words")
print(f"   - Growing pages (up): median {wc_by_trend['up']:.0f} words")
print("   → 'Down' and 'up' pages have almost identical word counts.")
print("   → This suggests we need more sophisticated signals than just 'longer content'.")

print("\n" + "=" * 60)
print("These numbers show that ML can add value beyond simple rules,")
print("and that there are real, measurable signals to work with.")
print("=" * 60)

SUPPORTING EVIDENCE FOR LANE 2: REFRESH / CONTENT OPPORTUNITY SCORING

1. Learned model beats hand-written rule:
   - Baseline (hand-written rule) Precision@50: 0.240 (~12 of top 50 right)
   - Random Forest Precision@50: 0.740 (~37 of top 50 right)
   → The learned model is ~3.1x more accurate at identifying pages worth reviewing.

2. CTR drops sharply as position worsens:
   - page_1: 0.3548% CTR
   - top_3: 0.3341% CTR
   - striking: 0.2558% CTR
   - page_3_5: 0.1424% CTR
   - deep: 0.0554% CTR
   → Position is a strong signal for engagement opportunity.

3. Content length alone doesn't explain decline:
   - Declining pages (down): median 2909 words
   - Growing pages (up): median 2848 words
   → 'Down' and 'up' pages have almost identical word counts.
   → This suggests we need more sophisticated signals than just 'longer content'.

These numbers show that ML can add value beyond simple rules,
and that there are real, measurable signals to work with.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What This Work Can Claim

This work produces **observed, directional, decision-support** findings. Specifically:

- We can claim that **a learned model (random forest) outperforms a hand-written rule** on the starter dataset, as measured by Precision@50 (0.740 vs 0.240). This suggests that ML can add value in ranking pages for review.

- We can claim that **position is strongly associated with CTR** – pages in better positions tend to have higher click-through rates. This is an observed pattern in the data.

- We can claim that **content length alone does not explain decline** – declining and growing pages have similar median word counts. This suggests that simple heuristics are insufficient.

- We can claim that **the system produces a ranked queue** that a content editor can use to prioritize their review work.

- We can claim that **reason codes provide transparency** – they explain why each page was flagged, which helps editors trust and verify the recommendations.

---

### What This Work Cannot Claim

This work **cannot** claim the following:

- **Causality.** We cannot claim that refreshing a page *causes* it to recover. Our analysis is observational – we observed patterns in historical data, but we did not run an experiment.

- **"Predicting Google."** We cannot claim to predict Google's algorithm, rankings, or any search engine behavior. We are analyzing observed signals from search data.

- **Universal truth.** The results come from a specific starter dataset (30,000 rows, 32 clients) and may not generalize to all content, all clients, or all time periods.

- **Optimality.** We cannot claim that our model or baseline is the "best" possible approach. We are comparing a specific rule and a specific model on a specific dataset.

- **A refresh will definitely help.** Even if a page is flagged as declining, we cannot guarantee that editing it will recover its traffic. The ranking is about *priority* – which pages to review first – not about guaranteed outcomes.

---

### The Core Distinction

> **We are building a decision-support tool, not a causal model.**
>
> The output is a **ranked queue** that helps humans spend their time more effectively. The model suggests which pages *might* benefit from review, but the final decision – and the responsibility – remains with the human editor.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.